# Kaggle smoke test — black-hole-ml

Proves the round trip on Kaggle: clone the repo, build the C++ core, run the
tests, install the Python package, and import it. This is Phase 0 plumbing —
no physics yet.

**Before running:** turn **Internet = On** in the notebook settings (right panel)
so `git clone` and `pip` can reach the network. GPU is *not* needed for this.


In [ ]:
# 1. Clone the repo into the writable working dir (idempotent)
import os, shutil
REPO_URL  = "https://github.com/amark-23/black-hole-ml.git"
CLONE_DIR = "/kaggle/working/black-hole-ml"
if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)
!git clone --depth 1 {REPO_URL} {CLONE_DIR}
%cd {CLONE_DIR}


In [ ]:
# 2. Make sure the C++ toolchain is present (cmake + g++)
!cmake --version || pip -q install cmake
!g++ --version | head -1


In [ ]:
# 3. Configure, build, and test the C++ core (CPU only; CUDA stays off)
!cmake -S sim -B sim/build -DCMAKE_BUILD_TYPE=Release
!cmake --build sim/build --parallel
!ctest --test-dir sim/build --output-on-failure


In [ ]:
# 4. Install the Python package (pulls numpy / pyyaml / matplotlib / imageio;
#    torch is already present on Kaggle)
!pip -q install -e .


In [ ]:
# 5. Confirm the package imports
import bhml
print("bhml imported OK from", bhml.__file__)


## Notes

- **Private repo?** Add a GitHub token as a Kaggle Secret and clone with it:
  `https://<TOKEN>@github.com/amark-23/black-hole-ml.git`. The repo is public for
  now, so the plain URL works.
- **GPU work** (ray tracing, training) comes in later phases; this notebook is
  CPU-only and costs no GPU quota.
- Re-running the first cell wipes and re-clones, so the notebook always tests a
  fresh checkout.
